# RAG local con Ollama + Weaviate

Notebook reorganizado a partir del experimento original. El objetivo es que cada etapa tenga una prueba y una validación antes de continuar.

```text
PDF → páginas → chunks → embeddings → Weaviate
                                      ↓
pregunta → embedding → búsqueda → contexto → Gemma → respuesta
```

Modelos actuales: `qwen3-embedding:0.6b` para embeddings y `gemma3:270m` para generación.



1. Configuración e imports
2. Validar Weaviate y Ollama
3. Cargar PDFs
4. Validar páginas y metadata
5. Crear chunks
6. Validar chunks
7. Crear colección
8. Probar embedding
9. Probar inserción y lectura
10. Probar embeddings por lote
11. Reiniciar colección
12. Indexar 1498 chunks
13. Validar indexación
14. Probar búsqueda semántica
15. Construir contexto
16. Probar generación
17. Ejecutar RAG completo



## 1. Configuración

In [1]:
from pathlib import Path

EMBEDDING_MODEL = "qwen3-embedding:0.6b"
GENERATION_MODEL = "gemma3:270m"
COLLECTION_NAME = "Document"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
RETRIEVAL_LIMIT = 3

# El notebook puede ejecutarse desde la raíz del proyecto o desde src/.
cwd = Path.cwd()
candidates = [
    cwd / "data" / "documents",
    cwd.parent / "data" / "documents",
]
DOCUMENTS_DIRECTORY = next(
    (path for path in candidates if path.exists()),
    candidates[0],
)

print("Directorio de documentos:")
print(DOCUMENTS_DIRECTORY.resolve())


Directorio de documentos:
/Users/daniel/Documents/agentic_AI/rag_test/data/documents


## 2. Imports

In [2]:
import ollama
import weaviate

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from weaviate.classes.config import Configure, DataType, Property


## 3. Validar el entorno

In [3]:
# VALIDACIÓN: Weaviate
with weaviate.connect_to_local() as client:
    assert client.is_ready(), "Weaviate no está disponible."
print("OK: Weaviate está listo.")


OK: Weaviate está listo.


In [4]:
# VALIDACIÓN: modelos de Ollama
models = ollama.list()
model_names = [model.model for model in models.models]
print("Modelos disponibles:")
for name in model_names:
    print(" -", name)
assert EMBEDDING_MODEL in model_names, f"No se encontró {EMBEDDING_MODEL}"
assert GENERATION_MODEL in model_names, f"No se encontró {GENERATION_MODEL}"
print("OK: modelos disponibles.")


Modelos disponibles:
 - gemma3:270m
 - qwen3-embedding:0.6b
 - qwen2.5-coder:7b
 - gemma4:e2b
 - qwen3-embedding:8b
 - mxbai-embed-large:latest
 - qwen3.5:0.8b
 - mistral:latest
OK: modelos disponibles.


## 4. Carga de PDFs

Cada página se conserva como un documento independiente para mantener `source`, `document_id` y `page`.

In [5]:
def load_documents():
    """Carga todos los PDFs y conserva metadata por página."""
    documents = []
    pdf_files = sorted(DOCUMENTS_DIRECTORY.glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(f"No se encontraron PDFs en {DOCUMENTS_DIRECTORY.resolve()}")

    print(f"PDFs encontrados: {len(pdf_files)}")

    for pdf_path in pdf_files:
        print(f"Procesando: {pdf_path.name}")
        reader = PdfReader(pdf_path)

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text()
            if not text or not text.strip():
                continue
            documents.append({
                "text": text.strip(),
                "source": pdf_path.name,
                "document_id": pdf_path.stem,
                "page": page_number,
            })

    print(f"\nPáginas cargadas: {len(documents)}")
    return documents


## 5. Validación de carga

En la ejecución anterior se obtuvieron **4 PDFs y 375 páginas con texto**.

In [6]:
documents = load_documents()
sources = sorted({d["source"] for d in documents})

print("PDFs:", len(sources))
print("Páginas:", len(documents))

assert len(sources) == 4
assert len(documents) == 375
assert documents[0]["text"].strip()

print("\nPRIMER DOCUMENTO")
print("Source:", documents[0]["source"])
print("Document ID:", documents[0]["document_id"])
print("Page:", documents[0]["page"])
print("Texto:")
print(documents[0]["text"][:2000])
print("\nOK: carga validada.")


PDFs encontrados: 4
Procesando: 9789240073593-eng.pdf
Procesando: 9789240073616-eng.pdf
Procesando: 9789240073630-eng.pdf
Procesando: 9789240073654-eng.pdf

Páginas cargadas: 375
PDFs: 4
Páginas: 375

PRIMER DOCUMENTO
Source: 9789240073593-eng.pdf
Document ID: 9789240073593-eng
Page: 1
Texto:
Carbohydrate intake  
for adults and children
WHO guideline

OK: carga validada.


## 6. Creación de chunks

In [7]:
def create_chunks(documents):
    """Divide cada página en chunks y conserva su metadata."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = []
    for document in documents:
        split_texts = text_splitter.split_text(document["text"])
        for chunk_index, text in enumerate(split_texts):
            chunks.append({
                "text": text,
                "source": document["source"],
                "document_id": document["document_id"],
                "page": document["page"],
                "chunk_index": chunk_index,
            })

    print(f"Chunks creados: {len(chunks)}")
    return chunks


## 7. Validación de chunks

El resultado conocido del experimento es **1498 chunks**. También validamos la metadata del primer chunk.

In [8]:
chunks = create_chunks(documents)
first_chunk = chunks[0]

assert len(chunks) == 1498
assert {"text", "source", "document_id", "page", "chunk_index"}.issubset(first_chunk)
assert first_chunk["text"].strip()

print("Source:", first_chunk["source"])
print("Document ID:", first_chunk["document_id"])
print("Page:", first_chunk["page"])
print("Chunk index:", first_chunk["chunk_index"])
print("Texto:")
print(first_chunk["text"])
print("\nOK: chunks validados.")


Chunks creados: 1498
Source: 9789240073593-eng.pdf
Document ID: 9789240073593-eng
Page: 1
Chunk index: 0
Texto:
Carbohydrate intake  
for adults and children
WHO guideline

OK: chunks validados.


## 8. Crear la colección `Document`

Usamos `self_provided()` porque Ollama genera los vectores y Weaviate los almacena y busca.

In [9]:
def reset_collection():
    """Elimina y recrea Document. Es destructiva para esa colección."""
    with weaviate.connect_to_local() as client:
        if client.collections.exists(COLLECTION_NAME):
            client.collections.delete(COLLECTION_NAME)
            print(f"Colección '{COLLECTION_NAME}' eliminada.")

        client.collections.create(
            name=COLLECTION_NAME,
            vector_config=Configure.Vectors.self_provided(),
            properties=[
                Property(name="text", data_type=DataType.TEXT),
                Property(name="source", data_type=DataType.TEXT),
                Property(name="document_id", data_type=DataType.TEXT),
                Property(name="page", data_type=DataType.INT),
                Property(name="chunk_index", data_type=DataType.INT),
            ],
        )
        print(f"Colección '{COLLECTION_NAME}' creada.")

reset_collection()

with weaviate.connect_to_local() as client:
    assert client.collections.exists(COLLECTION_NAME)
print("OK: colección validada.")


Colección 'Document' eliminada.
Colección 'Document' creada.
OK: colección validada.


## 9. Prueba de un embedding

Validamos que `qwen3-embedding:0.6b` produzca un vector de **1024 dimensiones**.

In [10]:
embedding = ollama.embed(
    model=EMBEDDING_MODEL,
    input=first_chunk["text"],
)["embeddings"][0]

print("Modelo:", EMBEDDING_MODEL)
print("Dimensión:", len(embedding))
print("Primeros 5 valores:", embedding[:5])
assert len(embedding) == 1024
print("OK: embedding validado.")


Modelo: qwen3-embedding:0.6b
Dimensión: 1024
Primeros 5 valores: [-0.13335234, 0.014014653, -0.008049484, 0.022838876, 0.061388507]
OK: embedding validado.


## 10. Prueba de inserción de un chunk

In [11]:
with weaviate.connect_to_local() as client:
    collection = client.collections.use(COLLECTION_NAME)
    result = collection.data.insert(
        properties={
            "text": first_chunk["text"],
            "source": first_chunk["source"],
            "document_id": first_chunk["document_id"],
            "page": first_chunk["page"],
            "chunk_index": first_chunk["chunk_index"],
        },
        vector=embedding,
    )
print("Chunk insertado correctamente.")
print("ID:", result)


Chunk insertado correctamente.
ID: c392d2fb-d740-4ad6-8160-b8c9ce44a392


## 11. Validación de lectura desde Weaviate

In [12]:
with weaviate.connect_to_local() as client:
    collection = client.collections.use(COLLECTION_NAME)
    results = collection.query.fetch_objects(limit=10)

assert len(results.objects) == 1
stored = results.objects[0]
print("Objetos almacenados:", len(results.objects))
print("Metadata recuperada:", stored.properties)
assert stored.properties["source"] == first_chunk["source"]
assert stored.properties["page"] == first_chunk["page"]
print("OK: inserción y lectura validadas.")


Objetos almacenados: 1
Metadata recuperada: {'page': 1, 'document_id': '9789240073593-eng', 'text': 'Carbohydrate intake  \nfor adults and children\nWHO guideline', 'source': '9789240073593-eng.pdf', 'chunk_index': 0}
OK: inserción y lectura validadas.


## 12. Prueba de embeddings por lote

Comprobamos si Ollama acepta varios textos en una sola llamada. Esperamos `5 → 5 → 1024`.

In [13]:
test_texts = [chunk["text"] for chunk in chunks[:5]]
batch_response = ollama.embed(
    model=EMBEDDING_MODEL,
    input=test_texts,
)
batch_embeddings = batch_response["embeddings"]

print("Textos enviados:", len(test_texts))
print("Embeddings recibidos:", len(batch_embeddings))
print("Dimensión:", len(batch_embeddings[0]))

assert len(batch_embeddings) == 5
assert all(len(vector) == 1024 for vector in batch_embeddings)
print("OK: embeddings por lote funcionan.")


Textos enviados: 5
Embeddings recibidos: 5
Dimensión: 1024
OK: embeddings por lote funcionan.


## 13. Reinicio limpio antes de indexar

Las pruebas anteriores dejaron un objeto en `Document`. Lo eliminamos antes de la indexación real.

> No ejecutes esta celda después de indexar los 1498 chunks, salvo que quieras borrar la colección.

In [14]:
reset_collection()

Colección 'Document' eliminada.
Colección 'Document' creada.


## 14. Indexación completa por lotes

Con `batch_size=20`, 1498 chunks requieren aproximadamente 75 llamadas de embedding en vez de 1498.

In [15]:
def index_chunks(chunks, batch_size=20):
    """Genera embeddings por lotes e inserta los chunks en Weaviate."""
    total = len(chunks)

    with weaviate.connect_to_local() as client:
        collection = client.collections.use(COLLECTION_NAME)

        for start in range(0, total, batch_size):
            batch = chunks[start:start + batch_size]
            texts = [chunk["text"] for chunk in batch]

            response = ollama.embed(
                model=EMBEDDING_MODEL,
                input=texts,
            )
            embeddings = response["embeddings"]
            assert len(embeddings) == len(batch)

            for chunk, vector in zip(batch, embeddings):
                collection.data.insert(
                    properties={
                        "text": chunk["text"],
                        "source": chunk["source"],
                        "document_id": chunk["document_id"],
                        "page": chunk["page"],
                        "chunk_index": chunk["chunk_index"],
                    },
                    vector=vector,
                )

            indexed = min(start + batch_size, total)
            print(f"Indexed: {indexed}/{total} ({indexed / total:.1%})")

    print("\nIndexación terminada.")


In [16]:
index_chunks(chunks, batch_size=20)

Indexed: 20/1498 (1.3%)
Indexed: 40/1498 (2.7%)
Indexed: 60/1498 (4.0%)
Indexed: 80/1498 (5.3%)
Indexed: 100/1498 (6.7%)
Indexed: 120/1498 (8.0%)
Indexed: 140/1498 (9.3%)
Indexed: 160/1498 (10.7%)
Indexed: 180/1498 (12.0%)
Indexed: 200/1498 (13.4%)
Indexed: 220/1498 (14.7%)
Indexed: 240/1498 (16.0%)
Indexed: 260/1498 (17.4%)
Indexed: 280/1498 (18.7%)
Indexed: 300/1498 (20.0%)
Indexed: 320/1498 (21.4%)
Indexed: 340/1498 (22.7%)
Indexed: 360/1498 (24.0%)
Indexed: 380/1498 (25.4%)
Indexed: 400/1498 (26.7%)
Indexed: 420/1498 (28.0%)
Indexed: 440/1498 (29.4%)
Indexed: 460/1498 (30.7%)
Indexed: 480/1498 (32.0%)
Indexed: 500/1498 (33.4%)
Indexed: 520/1498 (34.7%)
Indexed: 540/1498 (36.0%)
Indexed: 560/1498 (37.4%)
Indexed: 580/1498 (38.7%)
Indexed: 600/1498 (40.1%)
Indexed: 620/1498 (41.4%)
Indexed: 640/1498 (42.7%)
Indexed: 660/1498 (44.1%)
Indexed: 680/1498 (45.4%)
Indexed: 700/1498 (46.7%)
Indexed: 720/1498 (48.1%)
Indexed: 740/1498 (49.4%)
Indexed: 760/1498 (50.7%)
Indexed: 780/1498 (52.1

## 15. Validación de la indexación completa

Esperamos una correspondencia exacta: **1498 chunks = 1498 objetos en Weaviate**.

In [17]:
with weaviate.connect_to_local() as client:
    collection = client.collections.use(COLLECTION_NAME)
    result = collection.aggregate.over_all()

total_objects = result.total_count
print("Chunks creados:", len(chunks))
print("Objetos en Weaviate:", total_objects)
assert total_objects == len(chunks)
print("OK: indexación completa validada.")


Chunks creados: 1498
Objetos en Weaviate: 1498
OK: indexación completa validada.


## 16. Búsqueda semántica

In [18]:
def search_documents(query, limit=RETRIEVAL_LIMIT):
    """Convierte la pregunta en vector y recupera chunks cercanos."""
    query_vector = ollama.embed(
        model=EMBEDDING_MODEL,
        input=query,
    )["embeddings"][0]

    assert len(query_vector) == 1024

    with weaviate.connect_to_local() as client:
        collection = client.collections.use(COLLECTION_NAME)
        results = collection.query.near_vector(
            near_vector=query_vector,
            limit=limit,
        )

    return results.objects


## 17. Prueba de retrieval

Primero verificamos los fragmentos recuperados; todavía no le pedimos nada a Gemma.

In [19]:
query = "What does the guideline say about carbohydrate intake?"
results = search_documents(query, limit=RETRIEVAL_LIMIT)

print("Pregunta:", query)
print("Resultados recuperados:", len(results))
assert len(results) == RETRIEVAL_LIMIT

for i, obj in enumerate(results, start=1):
    print("\n" + "=" * 60)
    print(f"RESULTADO {i}")
    print("=" * 60)
    print("Fuente:", obj.properties["source"])
    print("Página:", obj.properties["page"])
    print("Chunk:", obj.properties["chunk_index"])
    print("\nTexto:")
    print(obj.properties["text"])


Pregunta: What does the guideline say about carbohydrate intake?
Resultados recuperados: 3

RESULTADO 1
Fuente: 9789240073593-eng.pdf
Página: 1
Chunk: 0

Texto:
Carbohydrate intake  
for adults and children
WHO guideline

RESULTADO 2
Fuente: 9789240073593-eng.pdf
Página: 3
Chunk: 0

Texto:
Carbohydrate intake  
for adults and children
WHO guideline

RESULTADO 3
Fuente: 9789240073593-eng.pdf
Página: 16
Chunk: 2

Texto:
Chronic Diseases (32). It is intended to complement other WHO guidance on healthy diets, particularly the 
WHO guideline on free sugars intake (15). The recommendations in this guideline are intended for the general 
population of adults and children. Setting a recommended level of carbohydrate intake (i.e. the amount of 
carbohydrate that should be consumed as a percentage of overall energy intake) was not included in the 
updating of the guidance on carbohydrate intake because the amount of carbohydrate, as determined by 
the 2002 Joint WHO/FAO Expert Consultation, was 

## 18. Construcción del contexto

Incluimos documento y página para conservar trazabilidad dentro del prompt.

In [20]:
def build_context(results):
    """Convierte resultados de Weaviate en contexto para el LLM."""
    parts = []
    for i, obj in enumerate(results, start=1):
        parts.append(
            f"[SOURCE {i}]\n"
            f"Document: {obj.properties['source']}\n"
            f"Page: {obj.properties['page']}\n"
            f"Text:\n{obj.properties['text']}"
        )
    return "\n\n".join(parts)

context = build_context(results)
print(context)
assert context.strip()
print("\nOK: contexto construido.")


[SOURCE 1]
Document: 9789240073593-eng.pdf
Page: 1
Text:
Carbohydrate intake  
for adults and children
WHO guideline

[SOURCE 2]
Document: 9789240073593-eng.pdf
Page: 3
Text:
Carbohydrate intake  
for adults and children
WHO guideline

[SOURCE 3]
Document: 9789240073593-eng.pdf
Page: 16
Text:
Chronic Diseases (32). It is intended to complement other WHO guidance on healthy diets, particularly the 
WHO guideline on free sugars intake (15). The recommendations in this guideline are intended for the general 
population of adults and children. Setting a recommended level of carbohydrate intake (i.e. the amount of 
carbohydrate that should be consumed as a percentage of overall energy intake) was not included in the 
updating of the guidance on carbohydrate intake because the amount of carbohydrate, as determined by 
the 2002 Joint WHO/FAO Expert Consultation, was based on the percentage of energy intake remaining 
after accounting for empirically determined total fat and protein intakes (3

## 19. Generación con Gemma

In [21]:
def generate_response(context, query):
    """Genera una respuesta usando únicamente el contexto."""
    prompt = f"""
Answer the question using only the information provided in the context.

If the answer is not in the context, respond:
"I did not find enough information."

When possible, mention the document name and page number.

Context:
{context}

Question:
{query}
"""

    response = ollama.chat(
        model=GENERATION_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]


## 20. Prueba del generador

In [22]:
answer = generate_response(context=context, query=query)
print("Respuesta:")
print(answer)
assert answer.strip()
print("\nOK: generación validada.")


Respuesta:
The guideline says that carbohydrate intake for adults and children is recommended to be between 15% and 30% of total energy intake.

OK: generación validada.


## 21. RAG completo

Integramos retrieval → contexto → generación.

In [23]:
def run_rag(query, limit=RETRIEVAL_LIMIT):
    """Ejecuta el pipeline RAG completo."""
    results = search_documents(query, limit=limit)
    context = build_context(results)
    return generate_response(context=context, query=query)


In [24]:
answer = run_rag(query, limit=3)
print("Pregunta:")
print(query)
print("\n" + "=" * 60)
print("RESPUESTA RAG")
print("=" * 60)
print(answer)


Pregunta:
What does the guideline say about carbohydrate intake?

RESPUESTA RAG
The guideline says that the recommended carbohydrate intake for adults and children is based on the percentage of energy intake remaining after accounting for empirically determined total fat and protein intakes.
